[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/AcademiXBase/deep-learning-from-scratch/blob/dev/my_notebooks/ch04_02.ipynb)

In [ ]:
# Colab で実行している場合、リポジトリをクローンする
#!git clone -b dev https://github.com/AcademiXBase/deep-learning-from-scratch.git
#%cd deep-learning-from-scratch/my_notebooks

In [ ]:
import sys, os
sys.path.append(os.pardir)  # 親ディレクトリのファイルをインポートするための設定
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import ipywidgets as widgets
from IPython.display import display

# ゼロつくオリジナルのモジュールをインポート
from common.functions import sigmoid, softmax, sigmoid_grad, cross_entropy_error
from common.gradient import numerical_gradient
from dataset.mnist import load_mnist

## 数値微分

ニューラルネットワークの学習では、モデルが正しい出力を与えるように損失関数の値をできるだけ小さくすることを目的とし、重み $w$ やバイアス $b$ などのパラメータを少しずつ変化させていきます。この更新の方向と大きさは、損失関数を各パラメータで偏微分して求める勾配にもとづいて決めることができます。  

ここで、勾配とは損失関数がどの方向にどれだけ変化するかを示します。勾配の負の方向にパラメータを動かすことで損失を減らすことができます。また、このような方法は一般的に勾配降下法 (gradient descent) と呼ばれます。  

例えば関数 $f(x) = x^2$ について考えると、導関数は  

$$f'(x) = 2x$$

です。点 $x=3$ における傾きは $6$ になるため、この勾配の負の方向に少しだけ $x$ を変化させることで、関数の値 $f(x)$ を小さくすることができます。  

一般的な更新式は次のように書けます：  

$$x_{\text{new}} = x - \alpha f'(x)$$

ここで $\alpha$ は学習率 (learning rate) と呼ばれる正の定数で、1 回の更新の大きさを調整する役割があります。  

In [ ]:
def f(x):
    return x**2

def df(x):
    return 2*x

def tangent_line(x, x0):
    return df(x0) * (x - x0) + f(x0)

x = 3
alpha = 0.3
new_x = x - alpha * df(x)

x_vals = np.linspace(-10, 10, 100)
y_vals = f(x_vals)

fig, ax = plt.subplots()
ax.plot(x_vals, y_vals, color="C0", label="$f(x) = x^2$")
ax.plot(x_vals, tangent_line(x_vals, x), color="C1", label=f"Tangent Line at x={x}")

ax.scatter([x], [f(x)], color="C2", zorder=5)  # 接点
ax.scatter([new_x], [f(new_x)], color="C3", zorder=5)  # 新しい点
ax.annotate("", xy=(new_x, f(new_x)), xytext=(x, f(x)), zorder=6,
            arrowprops=dict(facecolor="black", shrink=0, width=1, headwidth=5))
ax.text(x+1, f(x)-5, f"$f'(x={(x)})={df(x)}$", color="black", fontsize=16)

ax.set_xlim(-10, 10)
ax.set_ylim(-20, 100)
ax.axhline(0, color="black",linewidth=0.5, ls="--")
ax.axvline(0, color="black",linewidth=0.5, ls="--")
ax.legend(fancybox=False, framealpha=1, edgecolor="black")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title(f"Tangent Line to f(x) = $x^2$ at x={x}", fontsize=14, pad=15)
ax.grid()

ニューラルネットワークのモデルや数学的な関数については関数形が非常に複雑であり、個々のパラメータについて手作業で微分を導出するのが困難な場合があります。  

例えば、単純な関数 $f(x)$ の微分 $f'(x)$ を式として明示的に求める場合、これは解析微分 (analytical differentiation) と呼ばれます。しかし、関数形が複雑であったり、各層の結合や活性化関数が多数重なったりして 手で微分を求めにくい場合には近似的に勾配を求める手法が必要になります。  

それが数値微分 (numerical differentiation) です。これは微分の定義にもとづき、非常に小さな差分を用いて  

$$f'(x) \approx \frac{f(x+h) - f(x)}{h}$$

のように勾配を近似的に求める手法です。  

Python では次のようにして数値微分を計算することができます。  

In [ ]:
def num_diff(f, x, h=1e-4):
    return (f(x+h) - f(x)) / h

In [ ]:
# --- 数値微分の実装と可視化 ---
def f(x):
    return x**2

# 解析的な導関数
def df(x):
    return 2*x

# プロット範囲の設定
x_min, x_max, x_step = -20, 20, 0.1
y_min, y_max = -50, 400

# f(x) の定義域
x_vals = np.arange(x_min, x_max + x_step, x_step)

# --- スライダーの作成 ---
h_slider = widgets.FloatLogSlider(
    value=10,
    base=10,
    min=-4,  # 10^-4 = 0.0001
    max=1.0,  # 10^1 = 10
    step=0.1,
    description="h:",
    continuous_update=True,
    readout_format=".4f"
)

x_slider = widgets.FloatSlider(
    value=5.0,
    min=x_min,
    max=x_max,
    step=x_step,
    description="x:",
    continuous_update=True,
    readout_format=".2f"
)

output = widgets.Output()

def plot_with_params(h, x):
    # 解析解
    a_exact = df(x)
    b_exact = f(x) - a_exact * x

    plt.figure(figsize=(8, 6))

    # 関数 f(x) のプロット
    plt.plot(x_vals, f(x_vals), color="C0", label="f(x)", linewidth=2)

    # 解析的接線
    plt.plot(x_vals, a_exact * x_vals + b_exact, color="C1", label=f"Exact (f\'(x)={a_exact:.4f})", linewidth=2)

    # 近似接線
    a = num_diff(f, x, h)
    b = f(x) - a * x
    plt.plot(x_vals, a * x_vals + b, color="C2", linestyle="--", label=f"Approx. (f\'(x)={a:.4f})", linewidth=2)

    # 接点と差分点
    plt.scatter(x, f(x), color="C1", zorder=5)  # 接点
    plt.scatter(x+h, f(x+h), color="C2", zorder=5)  # x+hの点

    # 垂線
    plt.vlines(x=x, ymin=y_min, ymax=f(x), color="black", linestyle=":", linewidth=1.5)  # 接点からの垂線
    plt.vlines(x=x+h, ymin=y_min, ymax=f(x+h), color="black", linestyle=":", linewidth=1.5)  # x+h の点からの垂線

    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title(f"Numerical Differentiation at x={x}, f(x)={f(x)}", fontsize=14, pad=15)
    plt.legend(fancybox=False, framealpha=1, edgecolor="black")
    plt.grid()
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.show()

# --- 図の表示 ---
widgets.interact(plot_with_params, h=h_slider, x=x_slider)

display(output)


先ほど実装した数値微分は、前進差分 (forward difference) と呼ばれる方法になります。数値微分にはこのほかにも後退差分 (backward difference) や中心差分 (central difference) といった手法があります。  

1. 前進差分
$$f'(x) \approx \frac{f(x+h) - f(x)}{h}$$

2. 後退差分
$$f'(x) \approx \frac{f(x) - f(x-h)}{h}$$

3. 中心差分
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

特に、中心差分は評価点の左右両方の点を使って勾配を近似するため、前進差分や後退差分に比べて一般的に高い精度で微分値を求めることができます。これは中心差分が誤差の次数の観点でより有利であるためです。  

それぞれの方法は、Python で次のように実装できます。  

In [ ]:
def num_fwd_diff(f, x, h=1e-4):
    return (f(x+h) - f(x)) / h

def num_bwd_diff(f, x, h=1e-4):
    return (f(x) - f(x-h)) / h

def num_cent_diff(f, x, h=1e-4):
    return (f(x+h) - f(x-h)) / (2*h)

In [ ]:
# --- 数値微分の実装と可視化 ---
def f(x):
    return x**2
    #return x**2 + 3*x + 5

# 解析的な導関数
def df(x):
    return 2*x
    #return 2*x + 3

# プロット範囲の設定
x_min, x_max, x_step = -20, 20, 0.1
y_min, y_max = -50, 400

# f(x) の定義域
x_vals = np.arange(x_min, x_max + x_step, x_step)

# --- スライダーの作成 ---
h_slider = widgets.FloatLogSlider(
    value=10,
    base=10,
    min=-4,  # 10^-4 = 0.0001
    max=1.0,  # 10^1 = 10
    step=0.1,
    description="h:",
    continuous_update=True,
    readout_format=".4f"
)

x_slider = widgets.FloatSlider(
    value=5.0,
    min=x_min,
    max=x_max,
    step=x_step,
    description="x:",
    continuous_update=True,
    readout_format=".2f"
)

output = widgets.Output()

def plot_with_params(h, x):
    # 解析解
    a_exact = df(x)
    b_exact = f(x) - a_exact*x

    plt.figure(figsize=(8, 6))

    # 関数 f(x) のプロット
    plt.plot(x_vals, f(x_vals), label="f(x)", color="C0", linewidth=2)

    # 解析的な接線
    plt.plot(x_vals, a_exact * x_vals + b_exact, color="C1", label=f"Exact (f\'(x)={a_exact:.4f})", linewidth=2)

    # 近似接線
    a_fwd = num_fwd_diff(f, x, h)
    b_fwd = f(x) - a_fwd * x
    plt.plot(x_vals, a_fwd * x_vals + b_fwd, color="C2", linestyle="--", label=f"Forward (f\'(x)={a_fwd:.4f})", linewidth=2)

    a_bwd = num_bwd_diff(f, x, h)
    b_bwd = f(x) - a_bwd * x
    plt.plot(x_vals, a_bwd * x_vals + b_bwd, color="C3", linestyle="--", label=f"Backward (f\'(x)={a_bwd:.4f})", linewidth=2)

    a_cent = num_cent_diff(f, x, h)
    b_cent = f(x) - a_cent * x
    plt.plot(x_vals, a_cent * x_vals + b_cent, color="C4", linestyle="-.", label=f"Central (f\'(x)={a_cent:.4f})", linewidth=2)

    # 接点と差分点
    plt.scatter(x, f(x), color="C1", zorder=5)  # 接点
    plt.scatter(x+h, f(x+h), color="C2", zorder=5)  # x+hの点
    plt.scatter(x-h, f(x-h), color="C3", zorder=5)  # x-hの点

    # 垂線
    plt.vlines(x=x, ymin=y_min, ymax=f(x), color="k", linestyle=":", linewidth=1.5, zorder=4)  # 接点からの垂線
    plt.vlines(x=x+h, ymin=y_min, ymax=f(x+h), color="k", linestyle=":", linewidth=1.5, zorder=4)  # x+h の点からの垂線
    plt.vlines(x=x-h, ymin=y_min, ymax=f(x-h), color="k", linestyle=":", linewidth=1.5, zorder=4)  # x-h の点からの垂線

    # 差分点を結ぶ線
    plt.plot([x-h, x+h], [f(x-h), f(x+h)], color="gray", linestyle=":", linewidth=1.5)

    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title(f"Numerical Differentiation at x={x}, f(x)={f(x)}", fontsize=14, pad=15)
    plt.legend(fancybox=False, framealpha=1, edgecolor="black")
    plt.grid()
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.show()

# --- 図の表示 ---
widgets.interact(plot_with_params, h=h_slider, x=x_slider)

display(output)


In [ ]:
# --- 数値微分の実装と可視化 ---
def f(x):
    return 0.1*x**3 - 2*x**2 + 5*x + 10

# 解析的な導関数
def df(x):
    return 0.3*x**2 - 4*x + 5

# プロット範囲の設定
x_min, x_max, x_step = -15, 25, 0.1
y_min, y_max = -200, 400

# f(x) の定義域
x_vals = np.arange(x_min, x_max + x_step, x_step)

# --- スライダーの作成 ---
h_slider = widgets.FloatLogSlider(
    value=10,
    base=10,
    min=-4,  # 10^-4 = 0.0001
    max=1.0,  # 10^1 = 10
    step=0.1,
    description="h:",
    continuous_update=True,
    readout_format=".4f"
)

x_slider = widgets.FloatSlider(
    value=5.0,
    min=x_min,
    max=x_max,
    step=x_step,
    description="x:",
    continuous_update=True,
    readout_format=".2f"
)

output = widgets.Output()

def plot_with_params(h, x):
    # 解析解
    a_exact = df(x)
    b_exact = f(x) - a_exact*x

    plt.figure(figsize=(8, 6))

    # 関数 f(x) のプロット
    plt.plot(x_vals, f(x_vals), label="f(x)", color="C0", linewidth=2)

    # 解析的な接線
    plt.plot(x_vals, a_exact * x_vals + b_exact, color="C1", label=f"Exact (f\'(x)={a_exact:.4f})", linewidth=2)

    # 近似接線
    a_fwd = num_fwd_diff(f, x, h)
    b_fwd = f(x) - a_fwd * x
    plt.plot(x_vals, a_fwd * x_vals + b_fwd, color="C2", linestyle="--", label=f"Forward (f\'(x)={a_fwd:.4f})", linewidth=2)

    a_bwd = num_bwd_diff(f, x, h)
    b_bwd = f(x) - a_bwd * x
    plt.plot(x_vals, a_bwd * x_vals + b_bwd, color="C3", linestyle="--", label=f"Backward (f\'(x)={a_bwd:.4f})", linewidth=2)

    a_cent = num_cent_diff(f, x, h)
    b_cent = f(x) - a_cent * x
    plt.plot(x_vals, a_cent * x_vals + b_cent, color="C4", linestyle="-.", label=f"Central (f\'(x)={a_cent:.4f})", linewidth=2)

    # 接点と差分点
    plt.scatter(x, f(x), color="C1", zorder=5)  # 接点
    plt.scatter(x+h, f(x+h), color="C2", zorder=5)  # x+hの点
    plt.scatter(x-h, f(x-h), color="C3", zorder=5)  # x-hの点

    # 垂線
    plt.vlines(x=x, ymin=y_min, ymax=f(x), color="k", linestyle=":", linewidth=1.5, zorder=4)  # 接点からの垂線
    plt.vlines(x=x+h, ymin=y_min, ymax=f(x+h), color="k", linestyle=":", linewidth=1.5, zorder=4)  # x+h の点からの垂線
    plt.vlines(x=x-h, ymin=y_min, ymax=f(x-h), color="k", linestyle=":", linewidth=1.5, zorder=4)  # x-h の点からの垂線

    # 差分点を結ぶ線
    plt.plot([x-h, x+h], [f(x-h), f(x+h)], color="gray", linestyle=":", linewidth=1.5)

    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.title(f"Numerical Differentiation at x={x}, f(x)={f(x)}", fontsize=14, pad=15)
    plt.legend(fancybox=False, framealpha=1, edgecolor="black")
    plt.grid()
    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.show()

# --- 図の表示 ---
widgets.interact(plot_with_params, h=h_slider, x=x_slider)

display(output)


これまで見てきた微分は一変数関数に対するものでした。しかし、ニューラルネットワークには大量のパラメータ (変数) が含まれており、複数の変数を持つ関数に対する微分が必要になります。このような微分を偏微分 (partial differentiation) と言います。  

偏微分では、複数の変数それぞれについて関数の変化率を考えます。これは一変数の微分と同様に、ある変数だけをほんの少し変化させたときに関数がどれだけ変わるかを表しています。  

例えば、  

$$f(x_0, x_1)=x_0^2+x_1^2$$

という関数について考えると、各変数に関する偏微分は  

$$\frac{\partial f}{\partial x_0}=2x_0,\qquad \frac{\partial f}{\partial x_1}=2x_1$$

のようになります。  

この考え方は一般化すると、ベクトル

$$\mathbf{x} = \begin{pmatrix}x_1\\x_2\\\vdots\\x_n\end{pmatrix}$$

を入力とする関数 $f(\mathbf{x})$ に対して、その偏微分をまとめたものを  

$$
\frac{\partial f(\mathbf{x})}{\partial \mathbf{x}}=
\begin{pmatrix}
\frac{\partial f(\mathbf{x})}{\partial x_1} &
\frac{\partial f(\mathbf{x})}{\partial x_2} &
\cdots &
\frac{\partial f(\mathbf{x})}{\partial x_n}
\end{pmatrix}
$$

のように書くことができます。このとき、入力 $\mathbf{x}$ と偏微分ベクトル $\frac{\partial f(\mathbf{x})}{\partial \mathbf{x}}$ は、同じ形状 (同じ次元) になります。  


In [ ]:
# 2乗和関数
def f(x):
    if x.ndim == 1:
        return np.sum(x**2)
    else:
        return np.sum(x**2, axis=1)

In [ ]:
def num_grad(f, x, h=1e-4):
    grad = np.zeros_like(x)  # 勾配配列の初期化

    # 各要素に対して微小変化を加えて勾配を計算
    for idx in range(x.size):
        tmp_val = x[idx]  # 微分する変数を取り出す

        # f(x+h) の計算
        x[idx] = float(tmp_val) + h
        fxh1 = f(x)

        # f(x-h) の計算
        x[idx] = float(tmp_val) - h
        fxh2 = f(x)

        # 中心差分で勾配を計算
        grad[idx] = (fxh1 - fxh2) / (2*h)

        x[idx] = tmp_val  # 値を元に戻す

    return grad

In [ ]:
# メッシュ
x0_vals = np.linspace(-5, 5, 15)
x1_vals = np.linspace(-5, 5, 15)
X0, X1 = np.meshgrid(x0_vals, x1_vals)
X_grid = np.stack([X0, X1], axis=-1)
F = f(X_grid.reshape(-1, 2)).reshape(X0.shape)

output = widgets.Output()

def plot_3d(elev, azim, x0, x1):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")

    ax.set_xlabel(r"$x_0$")
    ax.set_ylabel(r"$x_1$")
    ax.set_zlabel(r"$f(x_0, x_1)$")
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_zlim(-10, 50)

    # 曲面
    ax.plot_surface(
        X0, X1, F,
        cmap="viridis", alpha=0.8,
        linewidth=0, edgecolor="none"
    )

    # 等高線
    ax.contour(X0, X1, F, colors="black", offset=-10)

    # 接点
    x = np.array([x0, x1])
    fx = f(x.reshape(1, -1))[0]
    ax.scatter(x0, x1, fx, s=60, c="black")

    # 勾配
    grad = num_grad(f, x.copy())

    # x0 方向の接線
    b0 = fx - grad[0]*x0
    tangent_x0 = grad[0]*x0_vals + b0
    ax.plot(x0_vals, np.full_like(x0_vals, x1), tangent_x0,
            color="C2", linewidth=3, label=r"$\partial f / \partial x_0$")

    # x1 方向の接線
    b1 = fx - grad[1]*x1
    tangent_x1 = grad[1]*x1_vals + b1
    ax.plot(np.full_like(x1_vals, x0), x1_vals, tangent_x1,
            color="C3", linewidth=3, label=r"$\partial f / \partial x_1$")

    # 接平面
    tangent_plane = (
        grad[0]*(X0 - x0) +
        grad[1]*(X1 - x1) +
        fx
    )
    ax.plot_surface(X0, X1, tangent_plane, color="C1", alpha=0.5)

    # 視点の設定
    ax.view_init(elev=elev, azim=azim)

    ax.legend()
    fig.tight_layout()
    plt.show()

# 図の描画
widgets.interact(
    plot_3d,
    elev=widgets.IntSlider(value=15, min=0, max=90, step=1, description="elev"),
    azim=widgets.IntSlider(value=0, min=-180, max=180, step=1, description="azim"),
    x0=widgets.FloatSlider(value=1.0, min=-4.0, max=4.0, step=0.1, description="x0"),
    x1=widgets.FloatSlider(value=3.0, min=-4.0, max=4.0, step=0.1, description="x1")
)

display(output)


In [ ]:
# メッシュ
x0_vals = np.linspace(-5, 5, 15)
x1_vals = np.linspace(-5, 5, 15)
X0, X1 = np.meshgrid(x0_vals, x1_vals)
X_grid = np.stack([X0, X1], axis=-1)
F = f(X_grid.reshape(-1, 2)).reshape(X0.shape)

# 勾配を初期化
grad = np.zeros((X0.size, 2))

# カウントを初期化
i = 0

# 1組ずつ勾配を計算
for x0, x1 in zip(X0.flatten(), X1.flatten()):
    # 配列にまとめる
    x = np.array([x0, x1])

    # 勾配を計算
    grad[i] = num_grad(f, x)

    # カウント
    i += 1

# 勾配を作図
plt.figure(figsize=(8, 8)) # 図の設定
plt.contourf(X0, X1, F, alpha=0.5) # 対象の関数
plt.quiver(X0, X1, -grad[:, 0], -grad[:, 1]) # 勾配
plt.xlabel("$x_0$") # x軸ラベル
plt.ylabel("$x_1$") # y軸ラベル
plt.grid() # グリッド線
plt.xlim(-5, 5)
plt.ylim(-5, 5)
plt.title("$f(x) = x_0^2 + x_1^2$", fontsize=14, pad=15)
plt.show() # 表示

# 2 層のニューラルネットワーク

In [ ]:
class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 重みの初期化
        self.params = {}
        self.params["W1"] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params["b1"] = np.zeros(hidden_size)
        self.params["W2"] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params["b2"] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params["W1"], self.params["W2"]
        b1, b2 = self.params["b1"], self.params["b2"]

        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)

        return y

    # x:入力データ, t:教師データ
    def loss(self, x, t):
        y = self.predict(x)

        return cross_entropy_error(y, t)

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        t = np.argmax(t, axis=1)

        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy

    # x:入力データ, t:教師データ
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads["W1"] = numerical_gradient(loss_W, self.params["W1"])
        grads["b1"] = numerical_gradient(loss_W, self.params["b1"])
        grads["W2"] = numerical_gradient(loss_W, self.params["W2"])
        grads["b2"] = numerical_gradient(loss_W, self.params["b2"])

        return grads

    def gradient(self, x, t):
        W1, W2 = self.params["W1"], self.params["W2"]
        b1, b2 = self.params["b1"], self.params["b2"]
        grads = {}

        batch_num = x.shape[0]

        # forward
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)

        # backward
        dy = (y - t) / batch_num
        grads["W2"] = np.dot(z1.T, dy)
        grads["b2"] = np.sum(dy, axis=0)

        dz1 = np.dot(dy, W2.T)
        da1 = sigmoid_grad(a1) * dz1
        grads["W1"] = np.dot(x.T, da1)
        grads["b1"] = np.sum(da1, axis=0)

        return grads

In [ ]:
# データ読み込み
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)
network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num      = 10000
train_size     = x_train.shape[0]
batch_size     = 100
learning_rate  = 0.1
iter_per_epoch = max(train_size // batch_size, 1)

# 保存リスト
train_loss_list = []
test_loss_list  = []
train_acc_list  = []
test_acc_list   = []
iter_list       = []  # iteration index の記録

# 学習ループ
for i in range(1, iters_num + 1):

    # ミニバッチ
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    # 勾配
    grad = network.gradient(x_batch, t_batch)

    # パラメータ更新
    for key in ("W1", "b1", "W2", "b2"):
        network.params[key] -= learning_rate * grad[key]

    # train loss（iterationごと）
    train_loss = network.loss(x_batch, t_batch)
    train_loss_list.append(train_loss)

    # iteration ごと上書き出力
    print(f"\riter {i}/{iters_num} | train loss: {train_loss:.4f}", end="")

    # epoch ごとの評価
    if i % iter_per_epoch == 0:
        test_loss = network.loss(x_test, t_test)
        train_acc  = network.accuracy(x_train, t_train)
        test_acc   = network.accuracy(x_test, t_test)

        # 保存
        test_loss_list.append(test_loss)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        iter_list.append(i)

        # 上書き出力更新
        print(
            f"\riter {i}/{iters_num} | "
            f"train loss: {train_loss:.4f}, test loss: {test_loss:.4f}, "
            f"train acc: {train_acc:.4f}, test acc: {test_acc:.4f}",
            end=""
        )

print()  # 最終表示後に改行

# ===== 可視化 =====
epochs = np.arange(len(train_acc_list))
plt.figure(figsize=(14, 6))

# --- accuracy plot ---
plt.subplot(1, 2, 1)
plt.plot(epochs, train_acc_list, label="train acc")
plt.plot(epochs, test_acc_list, label="test acc", linestyle="--")
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.ylim(0, 1.0)
plt.legend()

# --- train loss (iteration) ---
plt.subplot(1, 2, 2)
plt.plot(np.arange(len(train_loss_list)), train_loss_list, label="train loss")
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("Train Loss vs Iteration")
plt.legend()

plt.show()

参考資料

* [数値微分【ゼロつく1のノート(数学)】][1]  
* [勾配【ゼロつく1のノート(数学)】][2]  
* [勾配法【ゼロつく1のノート(数学)】][3]  

[1]: https://www.anarchive-beta.com/entry/2020/06/18/180000  
[2]: https://www.anarchive-beta.com/entry/2020/06/19/180000  
[3]: https://www.anarchive-beta.com/entry/2021/08/15/203504  